In [ ]:
# Code Block 1: Notebook description
#Notebook description
# This notebook evaluates the performance of a portfolio of assets on a buy-and-hold basis.
# It focuses on how multiple assets interact within the portfolio, rather than assessing a specific mechanical trading strategy.
# buy and hold is a good benchmark for any trading strategy, so it is important to evaluate the performance of a portfolio independently of
# trading strategies we may want to implement on the individual assets.


In [ ]:
# Code Block 2: Load Libraries
# Load Libraries
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from IPython.display import display
from schwab.auth import easy_client
import os
import sys
from pathlib import Path
# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()


from Quantapp.data import yf as qa_yf
from Quantapp.data import get_schwab_portfolio_snapshot
from Quantapp.data.adapters import SCHWAB_OPTION_SYMBOL_PATTERN

from Quantapp.data import MacroDataClient
from Quantapp.secrets import load_project_env, require_secret

load_project_env()

qe = MacroDataClient()

In [ ]:
# Code Block 3: Define functions and classes
#Define functions & Classes
#takes a dict of portfolio and their total amounts and directional (short or long value), converts dict to weightings instead of absolute values
def create_weighted_portfolio(portfolio):
    total = sum(abs(amount) for amount in portfolio.values())
    return {ticker: (amount / total) * (1 if direction == 'long' else -1)
            for (ticker, amount), direction in zip(portfolio.items(), ['long' if amount >= 0 else 'short' for amount in portfolio.values()])
    }
def create_weight_dict(portfolio):
    total = sum(abs(amount) for amount in portfolio.values())
    return {ticker: amount / total for ticker, amount in portfolio.items()}

def create_equal_weighted_dict(tickers):
    n = len(tickers)

    if n == 0:
        return {}

    equal_weight = 1 / n
    return {ticker: equal_weight for ticker in tickers}

def normalize_yf_ticker(ticker):
    if not isinstance(ticker, str):
        return ticker

    return ticker.strip().replace('/', '-')

def build_yf_ticker_map(tickers):
    return {ticker: normalize_yf_ticker(ticker) for ticker in tickers}

def zscore_series(series):
    mean = series.mean()
    std = series.std(ddof=0)

    if std == 0 or np.isnan(std):
        return pd.Series(0.0, index=series.index)

    return (series - mean) / std

def z_score(series):
    mean = series.mean()
    std = series.std(ddof=0)

    if std == 0 or np.isnan(std):
        return pd.Series(0.0, index=series.index)

    z = (series - mean) / std
    return z.replace([np.inf, -np.inf], np.nan)

def sharpe_annualized(series):
    mean = series.mean()
    std = series.std(ddof=0)

    if std == 0 or np.isnan(std):
        return 0.0

    return (mean / std) * np.sqrt(252)


In [ ]:
# Code Block 4: Define parameters
#Define parameters
time_frame_week = 7
time_frame_short = 21
time_frame_mid = 50
time_frame_long = 200
selected_time_frame = time_frame_long
CLIENT_ID = require_secret("SCHWAB_CLIENT_ID")
APP_SECRET = require_secret("SCHWAB_APP_SECRET")
CALLBACK_URL = os.getenv("SCHWAB_CALLBACK_URL", "https://127.0.0.1:8182")

_token_path = os.getenv("SCHWAB_TOKEN_PATH")
TOKEN_PATH = Path(_token_path).expanduser() if _token_path else PROJECT_ROOT / "schwab_token.json"
if not TOKEN_PATH.is_absolute():
    TOKEN_PATH = PROJECT_ROOT / TOKEN_PATH
TOKEN_PATH.parent.mkdir(parents=True, exist_ok=True)

#callback
period = '20y'
interval = '1d'
benchmark_str = 'SPY'


In [ ]:
# Code Block 5: Login to Schwab client
from schwab.auth import easy_client

client = easy_client(
    api_key=CLIENT_ID,
    app_secret=APP_SECRET,
    callback_url=CALLBACK_URL,
    token_path=str(TOKEN_PATH),
)

print(f"Schwab client ready. Token cache: {TOKEN_PATH}")


In [ ]:
# Code Block 6: Retrieve account and market data
# Schwab account retrieval and position normalization live in Quantapp.data.
portfolio_snapshot = get_schwab_portfolio_snapshot(client)

account_information = portfolio_snapshot.account_information
acct_map = portfolio_snapshot.account_numbers
acct_hash = portfolio_snapshot.account_hash
acct = portfolio_snapshot.account
positions = portfolio_snapshot.raw_positions
positions_df = portfolio_snapshot.option_positions
option_sentiment = portfolio_snapshot.option_sentiment
net_direction = portfolio_snapshot.net_direction
organized_positions = portfolio_snapshot.organized_positions
invested_symbols = portfolio_snapshot.invested_symbols
net_invested_amounts = portfolio_snapshot.net_invested_amounts
total_margin = portfolio_snapshot.total_margin
option_pattern = SCHWAB_OPTION_SYMBOL_PATTERN

# Retrieve core market data for benchmark and portfolio.
benchmark_symbol = benchmark_str if 'benchmark_str' in globals() else 'SPY'
benchmark_data = qa_yf.Ticker(benchmark_symbol).history(period=period, interval=interval)
invested_symbol_map = build_yf_ticker_map(invested_symbols)

if invested_symbol_map:
    portfolio_data = qa_yf.download(
        tickers=list(invested_symbol_map.values()),
        period=period,
        interval=interval,
        auto_adjust=True,
        threads=True,
        progress=False,
    )
    portfolio_closing_prices = portfolio_data['Close']
    if isinstance(portfolio_closing_prices, pd.Series):
        portfolio_closing_prices = portfolio_closing_prices.to_frame(name=invested_symbols[0])
    else:
        portfolio_closing_prices = portfolio_closing_prices.rename(
            columns={yf_ticker: ticker for ticker, yf_ticker in invested_symbol_map.items()}
        )
else:
    portfolio_closing_prices = pd.DataFrame(index=benchmark_data.index)

benchmark_close = benchmark_data['Close']
portfolio_closing_prices.index = portfolio_closing_prices.index.tz_localize(None)
benchmark_close.index = benchmark_close.index.tz_localize(None)

net_direction
raw_prices = portfolio_closing_prices.copy()


In [ ]:
# Code Block 9: DTE ladder
# =========================
# 9) Options expiration ladder
# =========================
from Quantapp.visualization.views.portfolio_profile.performance_structure import plot_options_expiration_ladder

fig = plot_options_expiration_ladder(positions_df)
if fig is not None:
    fig.show()


In [ ]:
# Code Block 7: Option P/L at expiration and net cost basis
# Retrieve net cost basis for each option grouped by ticker
# net cost basis = net_quantity * average_price * 100 (per contract)
import importlib
from Quantapp.visualization.views.portfolio_profile.performance_structure import option_expiration_pl as option_expiration_pl_module

option_expiration_pl_module = importlib.reload(option_expiration_pl_module)
build_option_profit_loss_extremes_table = option_expiration_pl_module.build_option_profit_loss_extremes_table
display_option_expiration_pl_view = option_expiration_pl_module.display_option_expiration_pl_view

if not positions_df.empty:
    positions_df['net_cost_basis'] = positions_df['net_quantity'] * positions_df['average_price'] * 100
    net_cost_basis = positions_df.groupby('underlying')['net_cost_basis'].sum().rename('net_cost_basis')
    # display(net_cost_basis.to_frame())
else:
    net_cost_basis = pd.Series(dtype=float, name='net_cost_basis')
    print("No options positions found.")

available_option_underlyings = (
    sorted(positions_df['underlying'].dropna().unique().tolist())
    if not positions_df.empty else []
)
benchmark_label_for_beta = benchmark_str if 'benchmark_str' in globals() else 'SPY'
portfolio_latest_prices = (
    portfolio_closing_prices.ffill().iloc[-1].dropna().to_dict()
    if isinstance(portfolio_closing_prices, pd.DataFrame) and not portfolio_closing_prices.empty
    else {}
)

def _ticker_lookup_key(ticker):
    return ''.join(character for character in str(ticker).upper() if character.isalnum())

benchmark_current_price = (
    float(benchmark_close.ffill().iloc[-1])
    if isinstance(benchmark_close, pd.Series) and not benchmark_close.dropna().empty
    else np.nan
)
benchmark_returns_for_beta = (
    benchmark_close.pct_change().dropna()
    if isinstance(benchmark_close, pd.Series)
    else pd.Series(dtype=float)
)
asset_returns_for_beta = (
    portfolio_closing_prices.pct_change()
    if isinstance(portfolio_closing_prices, pd.DataFrame) and not portfolio_closing_prices.empty
    else pd.DataFrame()
)
underlying_beta_map = {}

if not asset_returns_for_beta.empty and not benchmark_returns_for_beta.empty:
    for underlying_name in asset_returns_for_beta.columns:
        aligned_returns = pd.concat(
            [
                asset_returns_for_beta[underlying_name].rename('asset'),
                benchmark_returns_for_beta.rename('benchmark'),
            ],
            axis=1,
        ).dropna()

        if len(aligned_returns) < 2 or aligned_returns['benchmark'].var() == 0:
            underlying_beta_map[underlying_name] = 1.0
            continue

        beta_value = aligned_returns['asset'].cov(aligned_returns['benchmark']) / aligned_returns['benchmark'].var()

        if pd.notna(beta_value) and np.isfinite(beta_value):
            underlying_beta_map[underlying_name] = float(beta_value)
        else:
            underlying_beta_map[underlying_name] = 1.0
else:
    underlying_beta_map = {underlying_name: 1.0 for underlying_name in available_option_underlyings}

if available_option_underlyings:
    option_expiration_pl_view = display_option_expiration_pl_view(
        positions_df,
        net_cost_basis=net_cost_basis,
        portfolio_latest_prices=portfolio_latest_prices,
        benchmark_current_price=benchmark_current_price,
        underlying_beta_map=underlying_beta_map,
        benchmark_label=benchmark_label_for_beta,
    )
    option_profit_loss_extremes_table = build_option_profit_loss_extremes_table(
        positions_df,
        portfolio_latest_prices=portfolio_latest_prices,
    )
    option_direction_sign = (
        option_profit_loss_extremes_table['direction']
        .map({'up': 1.0, 'down': -1.0, 'flat': 0.0, 'none': 0.0})
        .fillna(1.0)
        .rename('sign')
    )
    option_direction_sign_by_lookup_key = {
        _ticker_lookup_key(ticker): sign
        for ticker, sign in option_direction_sign.items()
    }
    # display(option_profit_loss_extremes_table)
else:
    option_profit_loss_extremes_table = pd.DataFrame(columns=['max_profit', 'max_loss', 'direction'])
    option_direction_sign = pd.Series(dtype=float, name='sign')
    option_direction_sign_by_lookup_key = {}


In [ ]:
# Code Block 8: Option leg coverage audit
# =========================
# 8) Verify every raw Schwab option leg is represented in positions_df
# =========================
raw_option_rows = []
regex_miss_rows = []

for position in positions:
    instrument = position.get('instrument', {})

    if instrument.get('assetType') != 'OPTION':
        continue

    raw_symbol = instrument.get('symbol', '')
    normalized_symbol = ''.join((raw_symbol or '').split())
    match = option_pattern.match(normalized_symbol)
    expiration = pd.NaT
    parsed_option_type = pd.NA
    parsed_strike = pd.NA

    if match:
        expiration = pd.to_datetime('20' + match.group('expiration'), format='%Y%m%d', errors='coerce')
        parsed_option_type = match.group('option_type')
        parsed_strike = int(match.group('strike')) / 1000

    row = {
        'raw_symbol': raw_symbol,
        'normalized_symbol': normalized_symbol,
        'underlying_symbol': instrument.get('underlyingSymbol'),
        'put_call': instrument.get('putCall'),
        'expiration': expiration,
        'parsed_option_type': parsed_option_type,
        'parsed_strike': parsed_strike,
        'long_quantity': position.get('longQuantity', 0.0),
        'short_quantity': position.get('shortQuantity', 0.0),
        'average_price': position.get('averagePrice', 0.0),
    }
    raw_option_rows.append(row)

    if not match:
        regex_miss_rows.append(dict(row, drop_reason='symbol_did_not_match_option_pattern'))

raw_option_df = pd.DataFrame(raw_option_rows)
parsed_option_df = positions_df.copy()

if raw_option_df.empty:
    # print('[Code Block 7] No raw option legs returned by Schwab for the selected account.')
    pass
else:
    raw_audit = (
        raw_option_df.groupby('normalized_symbol', dropna=False, as_index=False)
        .agg(
            raw_rows=('normalized_symbol', 'size'),
            raw_long_quantity=('long_quantity', 'sum'),
            raw_short_quantity=('short_quantity', 'sum'),
            raw_average_price=('average_price', 'first'),
            underlying_symbol=('underlying_symbol', 'first'),
            expiration=('expiration', 'first'),
            put_call=('put_call', 'first'),
        )
    )
    if parsed_option_df.empty:
        parsed_audit = pd.DataFrame(columns=[
            'symbol',
            'parsed_rows',
            'parsed_long_quantity',
            'parsed_short_quantity',
            'parsed_average_price',
        ])
    else:
        parsed_option_df['symbol'] = parsed_option_df['symbol'].astype(str)
        parsed_audit = (
            parsed_option_df.groupby('symbol', as_index=False)
            .agg(
                parsed_rows=('symbol', 'size'),
                parsed_long_quantity=('long_quantity', 'sum'),
                parsed_short_quantity=('short_quantity', 'sum'),
                parsed_average_price=('average_price', 'first'),
            )
        )
    audit = raw_audit.merge(parsed_audit, left_on='normalized_symbol', right_on='symbol', how='left')
    audit[['parsed_rows', 'parsed_long_quantity', 'parsed_short_quantity']] = audit[
        ['parsed_rows', 'parsed_long_quantity', 'parsed_short_quantity']
    ].fillna(0)
    audit['raw_rows'] = audit['raw_rows'].fillna(0)
    audit['row_match'] = audit['raw_rows'].astype(int) == audit['parsed_rows'].astype(int)
    audit['long_match'] = audit['raw_long_quantity'].round(6) == audit['parsed_long_quantity'].round(6)
    audit['short_match'] = audit['raw_short_quantity'].round(6) == audit['parsed_short_quantity'].round(6)
    audit['present_in_positions_df'] = audit[['row_match', 'long_match', 'short_match']].all(axis=1)
    raw_symbol_count = raw_option_df['normalized_symbol'].nunique()
    parsed_symbol_count = parsed_option_df['symbol'].nunique() if not parsed_option_df.empty else 0
    # print('[Code Block 7] Option Leg Coverage Audit')
    # print(f'- Accounts returned by Schwab: {len(acct_map):,}')
    # print(f'- Selected account hash: {acct_hash}')
    # print(f'- Raw option legs from Schwab: {len(raw_option_df):,}')
    # print(f'- Parsed option legs in positions_df: {len(parsed_option_df):,}')
    # print(f'- Distinct option symbols from Schwab: {raw_symbol_count:,}')
    # print(f'- Distinct option symbols in positions_df: {parsed_symbol_count:,}')
    # print(f'- Regex misses: {len(regex_miss_rows):,}')

    if audit['present_in_positions_df'].all():
        # print('Status: PASS - every raw Schwab option leg is represented in positions_df.')
        pass
    else:
        # print('Status: FAIL - some option legs are missing or quantities do not match.')
        # display(
        #     audit.loc[
        #         ~audit['present_in_positions_df'],
        #         [
        #             'underlying_symbol',
        #             'normalized_symbol',
        #             'expiration',
        #             'put_call',
        #             'raw_rows',
        #             'parsed_rows',
        #             'raw_long_quantity',
        #             'parsed_long_quantity',
        #             'raw_short_quantity',
        #             'parsed_short_quantity',
        #         ],
        #     ].sort_values(['underlying_symbol', 'expiration', 'normalized_symbol'])
        # )
        pass
    if regex_miss_rows:
        # print('Fix suggestion: some option symbols failed the OCC regex, so keep those legs using instrument fields even when regex parsing fails.')
        # display(pd.DataFrame(regex_miss_rows).sort_values(['underlying_symbol', 'raw_symbol']))
        pass

    if len(acct_map) > 1:
        # print('Fix suggestion: your code currently pulls acct_map[0]. If your broker UI is showing another account, choose the matching hash or loop through every account.')
        pass


In [ ]:
# Code Block 10: Asset-level signed and unsigned analytics
# =========================
# 10) Asset-level signed and unsigned analytics
# =========================
rolling_windows = (21, 50, 200)
benchmark_label = 'SPY'

def _rolling_sharpe(values, window):
    rolling_mean = values.rolling(window).mean()
    rolling_std = values.rolling(window).std(ddof=0)
    sharpe = rolling_mean.div(rolling_std).mul(np.sqrt(252))
    return sharpe.mask(rolling_std.eq(0), 0.0).replace([np.inf, -np.inf], np.nan)

def _apply_zscore(values):
    if isinstance(values, pd.Series):
        return z_score(values)

    return values.apply(z_score)

def _latest_sorted_snapshot(frame):
    valid = frame.dropna(how='all')

    if valid.empty:
        return pd.Series(dtype=float)

    return valid.iloc[-1].sort_values(ascending=False)

def _log_summary(label, values):
    if isinstance(values, pd.DataFrame):
        valid = values.dropna(how='all')

        if valid.empty:
            return f'- {label}: empty DataFrame'

        return (
            f'- {label}: DataFrame {valid.shape[0]:,} x {valid.shape[1]} '
            f'({valid.index.min():%Y-%m-%d} to {valid.index.max():%Y-%m-%d})'
        )
    valid = values.dropna()

    if valid.empty:
        return f'- {label}: empty Series'

    return f'- {label}: Series {valid.shape[0]:,} rows ({valid.index.min():%Y-%m-%d} to {valid.index.max():%Y-%m-%d})'

# Daily return streams: unsigned raw returns and signed tradable returns
daily_returns = portfolio_closing_prices.pct_change().dropna(how='all')
benchmark_daily_returns = benchmark_close.pct_change().dropna()
if 'option_direction_sign_by_lookup_key' not in globals():
    raise RuntimeError('Run Code Block 7 before this block so option direction signs are available.')

signs = pd.Series(
    {
        ticker: option_direction_sign_by_lookup_key.get(_ticker_lookup_key(ticker), 1.0)
        for ticker in portfolio_closing_prices.columns
    },
    dtype=float,
)
daily_returns_signed = daily_returns.mul(signs, axis=1)
benchmark_daily_returns_aligned = benchmark_daily_returns.reindex(daily_returns.index).dropna()
daily_returns_for_corr = daily_returns.reindex(benchmark_daily_returns_aligned.index)
daily_returns_signed_for_corr = daily_returns_signed.reindex(benchmark_daily_returns_aligned.index)
portfolio_daily_returns_for_corr = daily_returns_signed.mean(axis=1).reindex(benchmark_daily_returns_aligned.index)
# print(f'[Code Block 9] Calculated daily return streams for {len(portfolio_closing_prices.columns)} assets.')

# Assets / Rolling horizon returns (unsigned + signed)
asset_return_windows = {
    window: portfolio_closing_prices.pct_change(window).dropna(how='all')
    for window in rolling_windows
}
asset_signed_return_windows = {
    window: frame.mul(signs, axis=1)
    for window, frame in asset_return_windows.items()
}
asset_return_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_return_windows.items()
}
asset_signed_return_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_signed_return_windows.items()
}
portfolio_assets_rolling_returns_21 = asset_return_windows[21]
portfolio_assets_rolling_returns_50 = asset_return_windows[50]
portfolio_assets_rolling_returns_200 = asset_return_windows[200]
portfolio_assets_rolling_signed_returns_21 = asset_signed_return_windows[21]
portfolio_assets_rolling_signed_returns_50 = asset_signed_return_windows[50]
portfolio_assets_rolling_signed_returns_200 = asset_signed_return_windows[200]
portfolio_assets_rolling_returns_z_scores_21 = asset_return_z_windows[21]
portfolio_assets_rolling_returns_z_scores_50 = asset_return_z_windows[50]
portfolio_assets_rolling_returns_z_scores_200 = asset_return_z_windows[200]
portfolio_assets_rolling_signed_return_z_scores_21 = asset_signed_return_z_windows[21]
portfolio_assets_rolling_signed_return_z_scores_50 = asset_signed_return_z_windows[50]
portfolio_assets_rolling_signed_return_z_scores_200 = asset_signed_return_z_windows[200]
# print('[Code Block 9] Calculated asset rolling returns and signed return z-scores for 21/50/200-day windows.')

# Assets / Rolling Sharpe (unsigned + signed)
asset_sharpe_windows = {
    window: _rolling_sharpe(daily_returns, window)
    for window in rolling_windows
}
asset_signed_sharpe_windows = {
    window: _rolling_sharpe(daily_returns_signed, window)
    for window in rolling_windows
}
asset_sharpe_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_sharpe_windows.items()
}
asset_signed_sharpe_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_signed_sharpe_windows.items()
}
portfolio_assets_rolling_sharpe_21 = asset_sharpe_windows[21]
portfolio_assets_rolling_sharpe_50 = asset_sharpe_windows[50]
portfolio_assets_rolling_sharpe_200 = asset_sharpe_windows[200]
portfolio_assets_rolling_signed_sharpe_21 = asset_signed_sharpe_windows[21]
portfolio_assets_rolling_signed_sharpe_50 = asset_signed_sharpe_windows[50]
portfolio_assets_rolling_signed_sharpe_200 = asset_signed_sharpe_windows[200]
portfolio_assets_rolling_sharpe_z_scores_21 = asset_sharpe_z_windows[21]
portfolio_assets_rolling_sharpe_z_scores_50 = asset_sharpe_z_windows[50]
portfolio_assets_rolling_sharpe_z_scores_200 = asset_sharpe_z_windows[200]
portfolio_assets_rolling_signed_sharpe_z_scores_21 = asset_signed_sharpe_z_windows[21]
portfolio_assets_rolling_signed_sharpe_z_scores_50 = asset_signed_sharpe_z_windows[50]
portfolio_assets_rolling_signed_sharpe_z_scores_200 = asset_signed_sharpe_z_windows[200]
# print('[Code Block 9] Calculated vectorized asset rolling Sharpe series for 21/50/200-day windows.')

# Assets / Rolling correlation to benchmark (unsigned + signed)
asset_corr_windows = {
    window: daily_returns_for_corr.rolling(window).corr(benchmark_daily_returns_aligned)
    for window in rolling_windows
}
asset_signed_corr_windows = {
    window: daily_returns_signed_for_corr.rolling(window).corr(benchmark_daily_returns_aligned)
    for window in rolling_windows
}
asset_corr_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_corr_windows.items()
}
asset_signed_corr_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_signed_corr_windows.items()
}
portfolio_assets_rolling_correlation_21 = asset_corr_windows[21]
portfolio_assets_rolling_correlation_50 = asset_corr_windows[50]
portfolio_assets_rolling_correlation_200 = asset_corr_windows[200]
portfolio_assets_rolling_signed_correlation_21 = asset_signed_corr_windows[21]
portfolio_assets_rolling_signed_correlation_50 = asset_signed_corr_windows[50]
portfolio_assets_rolling_signed_correlation_200 = asset_signed_corr_windows[200]
portfolio_assets_rolling_correlation_z_scores_21 = asset_corr_z_windows[21]
portfolio_assets_rolling_correlation_z_scores_50 = asset_corr_z_windows[50]
portfolio_assets_rolling_correlation_z_scores_200 = asset_corr_z_windows[200]
portfolio_assets_rolling_signed_correlation_z_scores_21 = asset_signed_corr_z_windows[21]
portfolio_assets_rolling_signed_correlation_z_scores_50 = asset_signed_corr_z_windows[50]
portfolio_assets_rolling_signed_correlation_z_scores_200 = asset_signed_corr_z_windows[200]
# print(f'[Code Block 9] Calculated asset rolling correlations to {benchmark_label} for 21/50/200-day windows.')

# Assets / Latest z-scores by metric
latest_assets_return_z_scores_21 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_return_z_scores_21)
latest_assets_return_z_scores_50 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_return_z_scores_50)
latest_assets_return_z_scores_200 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_return_z_scores_200)
latest_assets_sharpe_z_scores_21 = _latest_sorted_snapshot(portfolio_assets_rolling_sharpe_z_scores_21)
latest_assets_sharpe_z_scores_50 = _latest_sorted_snapshot(portfolio_assets_rolling_sharpe_z_scores_50)
latest_assets_sharpe_z_scores_200 = _latest_sorted_snapshot(portfolio_assets_rolling_sharpe_z_scores_200)
latest_assets_signed_sharpe_z_scores_21 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_sharpe_z_scores_21)
latest_assets_signed_sharpe_z_scores_50 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_sharpe_z_scores_50)
latest_assets_signed_sharpe_z_scores_200 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_sharpe_z_scores_200)

# Display labels: add parentheses only for signed series with negative direction
def format_ticker(ticker, sign):
    return f'({ticker})' if sign < 0 else ticker

latest_assets_return_z_scores_21.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_return_z_scores_21.index]
latest_assets_return_z_scores_50.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_return_z_scores_50.index]
latest_assets_return_z_scores_200.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_return_z_scores_200.index]
latest_assets_sharpe_z_scores_21.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_sharpe_z_scores_21.index]
latest_assets_sharpe_z_scores_50.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_sharpe_z_scores_50.index]
latest_assets_sharpe_z_scores_200.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_sharpe_z_scores_200.index]
latest_assets_signed_sharpe_z_scores_21.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_signed_sharpe_z_scores_21.index]
latest_assets_signed_sharpe_z_scores_50.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_signed_sharpe_z_scores_50.index]
latest_assets_signed_sharpe_z_scores_200.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_signed_sharpe_z_scores_200.index]

# Portfolio aggregates (signed equal-weight daily rebalance)
portfolio_daily_returns = daily_returns_signed.mean(axis=1)
portfolio_equity = (1 + portfolio_daily_returns).cumprod()
portfolio_return_21 = portfolio_equity.pct_change(21).dropna()
portfolio_return_50 = portfolio_equity.pct_change(50).dropna()
portfolio_return_200 = portfolio_equity.pct_change(200).dropna()
portfolio_rolling_sharpe_21 = _rolling_sharpe(portfolio_daily_returns, 21).dropna()
portfolio_rolling_sharpe_50 = _rolling_sharpe(portfolio_daily_returns, 50).dropna()
portfolio_rolling_sharpe_200 = _rolling_sharpe(portfolio_daily_returns, 200).dropna()

# Benchmark analytics (unsigned benchmark series)
benchmark_equity = (1 + benchmark_daily_returns).cumprod()
benchmark_rolling_sharpe_21 = _rolling_sharpe(benchmark_daily_returns, 21).dropna()
benchmark_rolling_sharpe_50 = _rolling_sharpe(benchmark_daily_returns, 50).dropna()
benchmark_rolling_sharpe_200 = _rolling_sharpe(benchmark_daily_returns, 200).dropna()
portfolio_rolling_signed_correlation_21 = portfolio_daily_returns_for_corr.rolling(21).corr(benchmark_daily_returns_aligned).dropna()
portfolio_rolling_signed_correlation_50 = portfolio_daily_returns_for_corr.rolling(50).corr(benchmark_daily_returns_aligned).dropna()
portfolio_rolling_signed_correlation_200 = portfolio_daily_returns_for_corr.rolling(200).corr(benchmark_daily_returns_aligned).dropna()
# print(f'[Code Block 9] Calculated portfolio and benchmark aggregate analytics versus {benchmark_label}.')
calculation_log = [
    '[Code Block 9] Summary:',
    f'- Assets analyzed: {len(portfolio_closing_prices.columns)}',
    f'- Rolling windows: {", ".join(str(window) for window in rolling_windows)} trading days',
    _log_summary('Asset daily returns', daily_returns),
    _log_summary('Signed asset daily returns', daily_returns_signed),
    _log_summary('Signed asset rolling returns (200d)', portfolio_assets_rolling_signed_returns_200),
    _log_summary('Asset rolling Sharpe (200d)', portfolio_assets_rolling_sharpe_200),
    _log_summary(f'Asset rolling correlation to {benchmark_label} (200d)', portfolio_assets_rolling_signed_correlation_200),
    _log_summary('Portfolio equity', portfolio_equity),
    _log_summary('Portfolio rolling Sharpe (200d)', portfolio_rolling_sharpe_200),
    _log_summary(f'Portfolio rolling correlation to {benchmark_label} (200d)', portfolio_rolling_signed_correlation_200),
]
# print("\n".join(calculation_log))


In [ ]:
# Code Block 11: Plots
# =========================
# 11) Plots
# =========================
from Quantapp.visualization.views.portfolio_profile.performance_structure import (
    plot_equity_curve,
    plot_rolling_correlation,
    plot_rolling_sharpe_zscore,
)

portfolio_equity_fig = plot_equity_curve(
    portfolio_equity,
    benchmark_equity,
    benchmark_label=benchmark_str,
)
portfolio_equity_fig.show()

rolling_sharpe_fig = plot_rolling_sharpe_zscore(
    {
        21: portfolio_rolling_sharpe_21,
        50: portfolio_rolling_sharpe_50,
        200: portfolio_rolling_sharpe_200,
    },
    {
        21: benchmark_rolling_sharpe_21,
        50: benchmark_rolling_sharpe_50,
        200: benchmark_rolling_sharpe_200,
    },
    benchmark_label=benchmark_str,
    default_window=200,
)
rolling_sharpe_fig.show()

rolling_correlation_fig = plot_rolling_correlation(
    {
        21: portfolio_rolling_signed_correlation_21,
        50: portfolio_rolling_signed_correlation_50,
        200: portfolio_rolling_signed_correlation_200,
    },
    benchmark_label=benchmark_str,
)
rolling_correlation_fig.show()


In [ ]:
# Code Block 12: Relative price strength z-score plots
# Relative Price Strength Z-Score Plots
from Quantapp.analytics.series_utils import calculate_zscore
from Quantapp.visualization.views.portfolio_profile.performance_structure import (
    format_snapshot_map,
    plot_benchmark_snapshot_zscores,
)

def risk_adjusted_returns(data, windows, ratio_type='sharpe', risk_free_rate=0.0, annualization_factor=252):
    if isinstance(windows, (str, bytes)):
        raise ValueError("windows must be an integer or an iterable of integers")
    try:
        window_list = [int(window) for window in windows]
    except TypeError:
        window_list = [int(windows)]
    if not window_list or any(window <= 0 for window in window_list):
        raise ValueError("windows must contain positive integers")

    price_frame = data.to_frame(name=data.name or "price") if isinstance(data, pd.Series) else data
    if not isinstance(price_frame, pd.DataFrame):
        raise TypeError("data must be a pandas Series or DataFrame")

    returns = price_frame.pct_change()
    if isinstance(risk_free_rate, pd.Series):
        periodic_rate = risk_free_rate.astype(float).sort_index().reindex(returns.index).ffill()
    elif np.isscalar(risk_free_rate):
        periodic_rate = pd.Series((1.0 + float(risk_free_rate)) ** (1.0 / annualization_factor) - 1.0, index=returns.index)
    else:
        raise TypeError("risk_free_rate must be a scalar annual rate or a pandas Series")

    excess_returns = returns.sub(periodic_rate, axis=0)
    single_window = len(window_list) == 1
    single_series = price_frame.shape[1] == 1
    output = []
    for column in returns.columns:
        excess = excess_returns[column]
        for window in window_list:
            mean_excess = excess.rolling(window).mean()
            if ratio_type == 'sharpe':
                volatility = excess.rolling(window).std()
                ratio = np.sqrt(annualization_factor) * mean_excess / volatility
                ratio = ratio.where(volatility > 0)
            elif ratio_type == 'sortino':
                downside = excess.where(excess < 0, 0.0)
                downside_deviation = downside.rolling(window).apply(lambda values: np.sqrt((values**2).mean()), raw=True)
                ratio = np.sqrt(annualization_factor) * mean_excess / downside_deviation
            else:
                raise ValueError("Invalid ratio_type. Use 'sharpe' or 'sortino'.")

            ratio = ratio.replace([np.inf, -np.inf], np.nan)
            ratio.name = f"{ratio_type}_ratio_{window}" if single_window and single_series else f"{column}_{ratio_type}_{window}"
            output.append(ratio)
    return pd.concat(output, axis=1)

time_frame_map = {
    '21': time_frame_short,
    '50': time_frame_mid,
    '200': time_frame_long,
}
if '_ticker_lookup_key' not in globals():
    def _ticker_lookup_key(ticker):
        return ''.join(character for character in str(ticker).upper() if character.isalnum())

if 'option_direction_sign_by_lookup_key' not in globals():
    raise RuntimeError('Run Code Block 7 before this block so option direction signs are available.')

sign_series = pd.Series(
    {
        ticker: option_direction_sign_by_lookup_key.get(_ticker_lookup_key(ticker), 1.0)
        for ticker in portfolio_closing_prices.columns
    },
    dtype=float,
)

asset_frame = portfolio_closing_prices.dropna(how='all').sort_index()
benchmark_series = benchmark_close.dropna().sort_index()
common_index = asset_frame.index.intersection(benchmark_series.index)
asset_frame = asset_frame.reindex(common_index).dropna(how='all')
benchmark_series = benchmark_series.reindex(common_index).dropna()
signed_returns = asset_frame.pct_change().mul(sign_series, axis=1)

def latest_zscore_snapshot(metric_frame):
    if metric_frame.empty:
        return pd.Series(dtype=float)
    zscore_frame = metric_frame.apply(calculate_zscore)
    if zscore_frame.empty:
        return pd.Series(dtype=float)
    return zscore_frame.iloc[-1].dropna().sort_values(ascending=False)

def latest_spread_zscore_snapshot(asset_metric_frame, benchmark_metric):
    if asset_metric_frame.empty:
        return pd.Series(dtype=float)
    benchmark_aligned = benchmark_metric.reindex(asset_metric_frame.index)
    spread_frame = asset_metric_frame.apply(lambda column: benchmark_aligned - column, axis=0)
    return latest_zscore_snapshot(spread_frame)

def rolling_sharpe_from_returns(returns, window, annualization_factor=252):
    rolling_mean = returns.rolling(window).mean()
    rolling_std = returns.rolling(window).std()
    ratio = np.sqrt(annualization_factor) * rolling_mean / rolling_std
    return ratio.where(rolling_std > 0).replace([np.inf, -np.inf], np.nan)

benchmark_snapshot = {
    'unsigned_asset_latest_zscores': {},
    'signed_asset_latest_zscores': {},
    'unsigned_spread_latest_zscores': {},
    'signed_spread_latest_zscores': {},
}

for term, window in time_frame_map.items():
    window = int(window)
    unsigned_sharpe = risk_adjusted_returns(
        asset_frame,
        windows=[window],
        ratio_type='sharpe',
    )
    if unsigned_sharpe.shape[1] == asset_frame.shape[1]:
        unsigned_sharpe.columns = asset_frame.columns

    benchmark_sharpe = risk_adjusted_returns(
        benchmark_series,
        windows=[window],
        ratio_type='sharpe',
    ).iloc[:, 0]
    signed_sharpe = rolling_sharpe_from_returns(signed_returns, window=window)

    benchmark_snapshot['unsigned_asset_latest_zscores'][term] = latest_zscore_snapshot(unsigned_sharpe)
    benchmark_snapshot['signed_asset_latest_zscores'][term] = latest_zscore_snapshot(signed_sharpe)
    benchmark_snapshot['unsigned_spread_latest_zscores'][term] = latest_spread_zscore_snapshot(
        unsigned_sharpe,
        benchmark_sharpe,
    )
    benchmark_snapshot['signed_spread_latest_zscores'][term] = latest_spread_zscore_snapshot(
        signed_sharpe,
        benchmark_sharpe,
    )
windows_signed = format_snapshot_map(
    benchmark_snapshot['signed_asset_latest_zscores'],
    sign_series,
)
windows_unsigned = benchmark_snapshot['unsigned_asset_latest_zscores']
windows_benchmark_minus_assets_signed = format_snapshot_map(
    benchmark_snapshot['signed_spread_latest_zscores'],
    sign_series,
)
windows_benchmark_minus_assets_unsigned = benchmark_snapshot['unsigned_spread_latest_zscores']

def set_dropdown_default(fig, target_label):
    if not fig.layout.updatemenus:
        return fig

    buttons = list(fig.layout.updatemenus[0].buttons)
    button_labels = [button.label for button in buttons]
    if target_label not in button_labels:
        return fig

    active_index = button_labels.index(target_label)
    active_button = buttons[active_index]
    visible = active_button.args[0].get('visible') if active_button.args else None
    if visible is not None:
        for trace, is_visible in zip(fig.data, visible):
            trace.visible = is_visible

    if len(active_button.args) > 1:
        title = active_button.args[1].get('title')
        if title:
            fig.update_layout(title=title)

    fig.layout.updatemenus[0].active = active_index
    return fig

snapshot_fig = plot_benchmark_snapshot_zscores(
    windows_signed=windows_signed,
    windows_unsigned=windows_unsigned,
    windows_benchmark_minus_assets_signed=windows_benchmark_minus_assets_signed,
    windows_benchmark_minus_assets_unsigned=windows_benchmark_minus_assets_unsigned,
    sign_series=sign_series,
    benchmark_label=benchmark_str,
)
snapshot_fig = set_dropdown_default(snapshot_fig, '200-Day')
snapshot_fig.show()


In [ ]:
# Code Block 13: Portfolio asset relationship clustering
# Compute raw-return correlations, distances, and hierarchical clustering inputs for Block 14.
from scipy.cluster.hierarchy import dendrogram, leaves_list, linkage
from scipy.spatial.distance import squareform

relationship_returns = (
    portfolio_closing_prices
    .pct_change()
    .replace([np.inf, -np.inf], np.nan)
    .dropna(how='all')
)
relationship_returns = relationship_returns.loc[:, relationship_returns.notna().sum() >= 2]
relationship_returns = relationship_returns.loc[:, relationship_returns.std(skipna=True) > 0]

if relationship_returns.shape[1] < 2:
    raise ValueError('At least two assets with valid return history are required for clustering.')

asset_correlation = relationship_returns.corr().clip(-1.0, 1.0)
asset_correlation = asset_correlation.dropna(how='all').dropna(axis=1, how='all')
asset_correlation = asset_correlation.fillna(0.0)
np.fill_diagonal(asset_correlation.values, 1.0)

asset_distance = (2.0 * (1.0 - asset_correlation)).clip(lower=0.0)
np.fill_diagonal(asset_distance.values, 0.0)
condensed_asset_distance = squareform(asset_distance.values, checks=False)
asset_linkage = linkage(condensed_asset_distance, method='average')
asset_order = asset_correlation.index[leaves_list(asset_linkage)].tolist()
clustered_asset_correlation = asset_correlation.loc[asset_order, asset_order]
clustered_asset_distance = asset_distance.loc[asset_order, asset_order]

dendro = dendrogram(
    asset_linkage,
    labels=asset_correlation.index.tolist(),
    no_plot=True,
)


In [ ]:
# Code Block 14: Benchmark-residual asset relationship clustering
# Regress each asset's daily returns against the benchmark, then cluster residual correlations.
from scipy.cluster.hierarchy import dendrogram, leaves_list, linkage
from scipy.spatial.distance import squareform
from plotly.subplots import make_subplots
import plotly.graph_objects as go

minimum_residual_observations = 30
residual_benchmark_label = benchmark_label if 'benchmark_label' in globals() else benchmark_str

benchmark_residual_asset_returns = (
    portfolio_closing_prices
    .pct_change()
    .replace([np.inf, -np.inf], np.nan)
    .dropna(how='all')
)
benchmark_residual_benchmark_returns = (
    benchmark_close
    .pct_change()
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)
benchmark_residual_benchmark_returns.name = residual_benchmark_label

common_residual_index = benchmark_residual_asset_returns.index.intersection(benchmark_residual_benchmark_returns.index)
benchmark_residual_asset_returns = benchmark_residual_asset_returns.reindex(common_residual_index)
benchmark_residual_benchmark_returns = benchmark_residual_benchmark_returns.reindex(common_residual_index)

def _benchmark_residualize(asset_returns, benchmark_returns, min_observations):
    residual_series_by_asset = {}
    regression_rows = []

    if benchmark_returns.std(skipna=True) == 0:
        raise ValueError('Benchmark returns must have non-zero variance for residual regression.')

    for ticker, asset_return_series in asset_returns.items():
        aligned_returns = pd.concat(
            [asset_return_series, benchmark_returns],
            axis=1,
            keys=['asset', 'benchmark'],
        ).dropna()

        if aligned_returns.shape[0] < min_observations:
            continue
        if aligned_returns['asset'].std(skipna=True) == 0 or aligned_returns['benchmark'].std(skipna=True) == 0:
            continue

        benchmark_values = aligned_returns['benchmark'].to_numpy(dtype=float)
        asset_values = aligned_returns['asset'].to_numpy(dtype=float)
        regression_matrix = np.column_stack([np.ones(len(aligned_returns)), benchmark_values])
        alpha, beta = np.linalg.lstsq(regression_matrix, asset_values, rcond=None)[0]
        fitted_values = alpha + beta * benchmark_values
        residual_series = pd.Series(
            asset_values - fitted_values,
            index=aligned_returns.index,
            name=ticker,
        )

        if residual_series.std(skipna=True) == 0:
            continue

        residual_series_by_asset[ticker] = residual_series
        regression_rows.append(
            {
                'Ticker': ticker,
                'Alpha': alpha,
                'Beta': beta,
                'Observations': aligned_returns.shape[0],
                'Residual Volatility': residual_series.std(ddof=0),
            }
        )

    residual_frame = pd.DataFrame(residual_series_by_asset).dropna(how='all')
    regression_stats = pd.DataFrame(regression_rows)
    if not regression_stats.empty:
        regression_stats = regression_stats.set_index('Ticker').sort_index()

    return residual_frame, regression_stats

benchmark_residual_returns, benchmark_residual_regression_stats = _benchmark_residualize(
    benchmark_residual_asset_returns,
    benchmark_residual_benchmark_returns,
    minimum_residual_observations,
)
benchmark_residual_returns = benchmark_residual_returns.loc[
    :,
    benchmark_residual_returns.notna().sum() >= minimum_residual_observations,
]

if benchmark_residual_returns.shape[1] < 2:
    raise ValueError(
        'At least two assets with valid benchmark-residual return history are required for clustering.'
    )

benchmark_residual_correlation = benchmark_residual_returns.corr(
    min_periods=minimum_residual_observations
).clip(-1.0, 1.0)
benchmark_residual_correlation = benchmark_residual_correlation.dropna(how='all').dropna(axis=1, how='all')
valid_residual_assets = benchmark_residual_correlation.index.intersection(benchmark_residual_correlation.columns)
benchmark_residual_correlation = benchmark_residual_correlation.loc[valid_residual_assets, valid_residual_assets]

if benchmark_residual_correlation.shape[1] < 2:
    raise ValueError(
        'At least two assets with overlapping benchmark-residual return history are required for clustering.'
    )

benchmark_residual_correlation = benchmark_residual_correlation.fillna(0.0)
np.fill_diagonal(benchmark_residual_correlation.values, 1.0)

benchmark_residual_distance = (2.0 * (1.0 - benchmark_residual_correlation)).clip(lower=0.0)
np.fill_diagonal(benchmark_residual_distance.values, 0.0)
condensed_residual_distance = squareform(benchmark_residual_distance.values, checks=False)
benchmark_residual_linkage = linkage(condensed_residual_distance, method='average')
benchmark_residual_order = benchmark_residual_correlation.index[leaves_list(benchmark_residual_linkage)].tolist()
clustered_benchmark_residual_correlation = benchmark_residual_correlation.loc[
    benchmark_residual_order,
    benchmark_residual_order,
]
clustered_benchmark_residual_distance = benchmark_residual_distance.loc[
    benchmark_residual_order,
    benchmark_residual_order,
]

benchmark_residual_dendro = dendrogram(
    benchmark_residual_linkage,
    labels=benchmark_residual_correlation.index.tolist(),
    no_plot=True,
)

required_raw_clustering_vars = [
    'dendro',
    'clustered_asset_correlation',
    'clustered_asset_distance',
]
missing_raw_clustering_vars = [name for name in required_raw_clustering_vars if name not in globals()]
if missing_raw_clustering_vars:
    raise RuntimeError('Run Code Block 13 before Code Block 14 so raw-return clustering inputs are available.')

relationship_comparison_fig = make_subplots(
    rows=2,
    cols=2,
    row_heights=[0.32, 0.68],
    column_widths=[0.5, 0.5],
    vertical_spacing=0.08,
    horizontal_spacing=0.08,
    subplot_titles=(
        'Raw Return Dendrogram',
        f'Benchmark-Residual Dendrogram ({residual_benchmark_label})',
        'Raw Return Correlation Heatmap',
        'Benchmark-Residual Correlation Heatmap',
    ),
)

for icoord, dcoord in zip(dendro['icoord'], dendro['dcoord']):
    relationship_comparison_fig.add_trace(
        go.Scatter(
            x=icoord,
            y=dcoord,
            mode='lines',
            line=dict(color='#38BDF8', width=1.5),
            hoverinfo='skip',
            showlegend=False,
        ),
        row=1,
        col=1,
    )

for icoord, dcoord in zip(benchmark_residual_dendro['icoord'], benchmark_residual_dendro['dcoord']):
    relationship_comparison_fig.add_trace(
        go.Scatter(
            x=icoord,
            y=dcoord,
            mode='lines',
            line=dict(color='#A78BFA', width=1.5),
            hoverinfo='skip',
            showlegend=False,
        ),
        row=1,
        col=2,
    )

raw_leaf_x = [5 + 10 * idx for idx in range(len(dendro['ivl']))]
residual_leaf_x = [5 + 10 * idx for idx in range(len(benchmark_residual_dendro['ivl']))]
relationship_comparison_fig.update_xaxes(
    tickmode='array',
    tickvals=raw_leaf_x,
    ticktext=dendro['ivl'],
    tickangle=-35,
    row=1,
    col=1,
)
relationship_comparison_fig.update_xaxes(
    tickmode='array',
    tickvals=residual_leaf_x,
    ticktext=benchmark_residual_dendro['ivl'],
    tickangle=-35,
    row=1,
    col=2,
)
relationship_comparison_fig.update_yaxes(title_text='Distance', row=1, col=1)
relationship_comparison_fig.update_yaxes(title_text='Distance', row=1, col=2)

relationship_comparison_fig.add_trace(
    go.Heatmap(
        z=clustered_asset_correlation.values,
        x=clustered_asset_correlation.columns,
        y=clustered_asset_correlation.index,
        zmin=-1,
        zmax=1,
        colorscale='RdBu',
        reversescale=True,
        colorbar=dict(title='Raw Correlation', x=0.46, y=0.31, len=0.56),
        customdata=clustered_asset_distance.values,
        hovertemplate=(
            'Asset X: %{x}<br>'
            'Asset Y: %{y}<br>'
            'Raw Correlation: %{z:.3f}<br>'
            'Distance: %{customdata:.3f}<extra></extra>'
        ),
    ),
    row=2,
    col=1,
)
relationship_comparison_fig.add_trace(
    go.Heatmap(
        z=clustered_benchmark_residual_correlation.values,
        x=clustered_benchmark_residual_correlation.columns,
        y=clustered_benchmark_residual_correlation.index,
        zmin=-1,
        zmax=1,
        colorscale='RdBu',
        reversescale=True,
        colorbar=dict(title='Residual Correlation', x=1.02, y=0.31, len=0.56),
        customdata=clustered_benchmark_residual_distance.values,
        hovertemplate=(
            'Asset X: %{x}<br>'
            'Asset Y: %{y}<br>'
            'Residual Correlation: %{z:.3f}<br>'
            'Distance: %{customdata:.3f}<extra></extra>'
        ),
    ),
    row=2,
    col=2,
)
relationship_comparison_fig.update_xaxes(title_text='Assets', tickangle=-35, row=2, col=1)
relationship_comparison_fig.update_xaxes(title_text='Assets', tickangle=-35, row=2, col=2)
relationship_comparison_fig.update_yaxes(title_text='Assets', autorange='reversed', row=2, col=1)
relationship_comparison_fig.update_yaxes(title_text='Assets', autorange='reversed', row=2, col=2)

relationship_comparison_fig.update_layout(
    title=f'Portfolio Asset Relationship Clustering: Raw Returns vs {residual_benchmark_label} Residuals',
    template='plotly_dark',
    height=1050,
    margin=dict(l=70, r=70, t=100, b=120),
)
relationship_comparison_fig.show()
